In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    max as spark_max,
)

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------
# 1. Load feature table
# ---------------------------------------------------------

features = spark.table(
    "northmart_dev.silver.fraud_features_5min"
)

# ---------------------------------------------------------
# 2. Load silver transactions
# ---------------------------------------------------------

transactions = spark.table(
    "northmart_dev.silver.fraud_transactions_silver"
)

# ---------------------------------------------------------
# 3. Create fraud label on transactions
# ---------------------------------------------------------

transactions_labeled = (
    transactions
    .withColumn(
        "is_fraud",
        when(col("fraud_scenario").isNotNull(), 1).otherwise(0)
    )
)

# ---------------------------------------------------------
# 4. Attach transactions to their 5-minute feature window
# ---------------------------------------------------------

joined = (
    features.alias("f")
    .join(
        transactions_labeled.alias("t"),
        (
            (col("f.card_id") == col("t.card_id"))
            & (col("t.event_time") >= col("f.window.start"))
            & (col("t.event_time") < col("f.window.end"))
        ),
        "left"
    )
)

# ---------------------------------------------------------
# 5. One ML observation per card / 5-minute window
# ---------------------------------------------------------

ml_dataset = (
    joined
    .groupBy(
        col("f.card_id").alias("card_id"),
        col("f.window").alias("window"),
        col("f.transaction_count_5min").alias(
            "transaction_count_5min"
        ),
        col("f.amount_sum_5min").alias(
            "amount_sum_5min"
        ),
    )
    .agg(
        spark_max(
            when(col("t.is_fraud").isNull(), 0)
            .otherwise(col("t.is_fraud"))
        ).alias("is_fraud")
    )
)

print("\n=== ML DATASET ===")

ml_dataset.printSchema()

ml_dataset.show(
    20,
    truncate=False
)

print("\n=== LABEL DISTRIBUTION ===")

ml_dataset.groupBy("is_fraud").count().show()


=== ML DATASET ===
root
 |-- card_id: string (nullable = true)
 |-- window: struct (nullable = true)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- transaction_count_5min: long (nullable = true)
 |-- amount_sum_5min: double (nullable = true)
 |-- is_fraud: integer (nullable = true)

+-----------+------------------------------------------+----------------------+---------------+--------+
|card_id    |window                                    |transaction_count_5min|amount_sum_5min|is_fraud|
+-----------+------------------------------------------+----------------------+---------------+--------+
|CARD-012243|{2026-08-21 16:05:00, 2026-08-21 16:10:00}|1                     |215.19         |0       |
|CARD-007820|{2026-08-21 16:05:00, 2026-08-21 16:10:00}|1                     |151.92         |0       |
|CARD-013528|{2026-08-21 16:05:00, 2026-08-21 16:10:00}|1                     |174.21         |0       |
|CARD-002563|{2026-08-21 14:35:00, 2026

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

features = spark.table(
    "northmart_dev.silver.fraud_features_5min"
)

transactions = spark.table(
    "northmart_dev.silver.fraud_transactions_silver"
)

print("\n=== FEATURES ===")
features.printSchema()
features.show(10, truncate=False)

print("\n=== TRANSACTIONS SILVER ===")
transactions.printSchema()
transactions.show(10, truncate=False)


=== FEATURES ===
root
 |-- window: struct (nullable = true)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- card_id: string (nullable = true)
 |-- transaction_count_5min: long (nullable = true)
 |-- amount_sum_5min: double (nullable = true)

+------------------------------------------+-----------+----------------------+---------------+
|window                                    |card_id    |transaction_count_5min|amount_sum_5min|
+------------------------------------------+-----------+----------------------+---------------+
|{2026-08-21 16:05:00, 2026-08-21 16:10:00}|CARD-012243|1                     |215.19         |
|{2026-08-21 15:50:00, 2026-08-21 15:55:00}|CARD-010589|1                     |108.4          |
|{2026-08-21 15:05:00, 2026-08-21 15:10:00}|CARD-012308|1                     |75.94          |
|{2026-08-21 14:35:00, 2026-08-21 14:40:00}|CARD-003912|1                     |76.52          |
|{2026-08-21 15:55:00, 2026-08-21 16:00: